# Jobbannonsanalys för AI-utvecklare

Detta notebook hämtar jobbannonser från JobTech API, sparar dem i CSV-format och analyserar hur många jobb som finns inom olika yrken, arbetsgivare och platser.


In [ ]:
# Importar nödvändiga bibliotek som csv, counter (för att räkna antal jobbannonser) och requests (för att hämta data från API:t)
import csv
from collections import Counter

import requests

API_URL = "https://jobsearch.api.jobtechdev.se/search?q="
CSV_FILE = "data.csv"
KEYWORDS = ["python", "ai", "sql", "azure", "aws", "machine learning"]

# Föräldrar klass för att representera en jobbanons.
class JobAnnons:
    def __init__(self, headline, employer, url, location):
        self.headline = headline
        self.employer = employer
        self.url = url
        self.location = location

    def __str__(self):
        return f"{self.headline} - {self.employer} ({self.location})"

# Barnklass för att representera en jobbannons med AI-fokus. Super används för att ärva attributen från JobAnnons-klassen. Dessutom läggs till ett nytt attribut ai_focus som indikerar om annonsen har AI-fokus eller inte.
class AI_Job_Annons(JobAnnons):
    def __init__(self, headline, employer, url, location, ai_focus=False):
        super().__init__(headline, employer, url, location)
        self.ai_focus = ai_focus

# API hämtar annonser. Try except är inskrivet för att fånga felmeddelanden.
def fetch_jobs(search_term, limit=20):
    url = API_URL + search_term
    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        data = response.json() 
        hits = data.get("hits", [])
        if not isinstance(hits, list):
            print("API:t returnerade ingen jobblista.")
            return []
        return hits[:limit]
    except requests.exceptions.RequestException as error:
        print(f"Nätverksfel: {error}")
        return []
    except ValueError as error:
        print(f"Fel i API-svaret: {error}")
        return []

# Omvandla rådata från API:t till objekt i Python. Headline, employer, url och location.
def make_job_objects(positions):
    jobs = []
    for job in positions:
        headline = job.get("headline") or "Okänd titel"
        employer = job.get("employer", {}).get("name") or "Okänd arbetsgivare"
        url = job.get("webpage_url") or "Ingen länk"
        location = job.get("workplace_address", {}).get("municipality") or "Okänd ort"

        text = (headline + " " + employer + " " + location).lower()
        ai_focus = any(word in text for word in ["ai", "machine learning", "python", "sql", "azure", "aws"])

        jobs.append(AI_Job_Annons(headline, employer, url, location, ai_focus))
    return jobs

# 
def save_to_csv(jobs, filename=CSV_FILE):
    try:
        with open(filename, "w", newline="", encoding="utf-8") as file:
            writer = csv.writer(file)
            writer.writerow(["headline", "employer", "location", "webpage_url", "ai_focus"])
            for job in jobs:
                writer.writerow([job.headline, job.employer, job.location, job.url, job.ai_focus])
        print(f"Data sparades i {filename}.")
    except OSError as error:
        print(f"Kunde inte skriva CSV: {error}")

# Räknar antalet jobb som finns per arbetsgivare eller plats.
def count_by_field(jobs, field_name):
    counter = Counter()
    for job in jobs:
        value = getattr(job, field_name)
        counter[value] += 1
    return counter

# Hur många annonser som innehåller vissa ord.
def count_keyword_hits(jobs):
    counter = Counter()
    for job in jobs:
        text = (job.headline + " " + job.employer + " " + job.location).lower()
        for keyword in KEYWORDS:
            if keyword in text:
                counter[keyword] += 1
    return counter


search_term = "AI-utvecklare"
positions = fetch_jobs(search_term, limit=20)

if not positions:
    print("Inga jobb hittades. Testa ett annat sökord.")
else:
    jobs = make_job_objects(positions)
    save_to_csv(jobs)

    print("\nTotalt antal jobb:", len(jobs))

    employer_count = count_by_field(jobs, "employer")
    print("\nVanligaste arbetsgivare:")
    for employer, count in employer_count.most_common(5):
        print(f"- {employer}: {count}")

    location_count = count_by_field(jobs, "location")
    print("\nVanligaste platser:")
    for place, count in location_count.most_common(5):
        print(f"- {place}: {count}")

    keyword_count = count_keyword_hits(jobs)
    print("\nVanliga sökord i annonserna:")
    for keyword, count in keyword_count.most_common(5):
        print(f"- {keyword}: {count}")


Data sparades i data.csv.

Totalt antal jobb: 10

Vanligaste arbetsgivare:
- Softhouse Nordic AB: 2
- Syntronic Aktiebolag: 2
- Intensogruppen AB: 1
- Vizzit International AB: 1
- NXT Interim Stockholm AB: 1

Vanligaste platser:
- Göteborg: 3
- Stockholm: 3
- Solna: 1
- Linköping: 1
- Okänd ort: 1

Vanliga sökord i annonserna:
- ai: 7
- python: 2
